# 01 · Event exploration

Load a cohort, inspect one event end to end, and exercise every export
surface the research layer offers.

> **This notebook is a research client, not a pipeline.** It contains no SQL, no
> threshold, and no classification rule. Every number comes from
> `afterhours_lab.research`, which reads persisted features computed once by
> `afterhours_lab.reactions`. Nothing here writes to the database — the pool is
> opened read-only.

In [ ]:
import datetime as dt

from afterhours_lab.research import (
    EventFilter,
    fetch_cohort,
    fetch_event_detail,
    fetch_class_distribution,
    fetch_monthly_counts,
    to_csv,
    to_jsonl,
    to_pandas,
    to_polars,
    write_parquet,
)
from afterhours_lab.research.notebook import (
    research_pool,
    describe_filter,
    describe_cohort,
    show_cohort,
    development_split,
)

# Jupyter already runs an event loop, so `await` works at cell top level.
pool = await research_pool()

## The cohort

State the `EventFilter` and its version triple explicitly, then disclose
how many events the filter excluded from the date/symbol universe.

In [ ]:
today = dt.date.today()
cohort_filter = EventFilter(
    date_from=today - dt.timedelta(days=180),
    date_to=today,
    order_by='earnings_date_desc',
    limit=200,
)
print(describe_filter(cohort_filter))

In [ ]:
async with pool.acquire() as conn:
    cohort = show_cohort(await fetch_cohort(conn, cohort_filter))

In [ ]:
df = to_pandas(cohort.rows)
df[['symbol', 'earnings_date', 'analysis_status', 'reaction_class',
    'initial_return', 'return_105m', 'retention']].head(20)

## One event in full

`fetch_event_detail` assembles the summary, the auditable coverage rows,
the exact bars those retrievals name, and any researcher notes in one call.

In [ ]:
if cohort.rows:
    pick = cohort.rows[0]
    async with pool.acquire() as conn:
        detail = await fetch_event_detail(conn, pick.symbol, pick.earnings_date)
    print(pick.symbol, pick.earnings_date)
else:
    detail = None
    print('cohort is empty for this window')

In [ ]:
detail.summary if detail else None

In [ ]:
if detail:
    display(to_pandas(detail.coverage))

In [ ]:
if detail:
    bars = to_pandas(detail.bars)
    print(len(bars), 'bars across phases:', sorted(bars['phase'].unique()) if len(bars) else [])
    display(bars.head())

In [ ]:
if detail:
    for note in detail.notes:
        print(f'[{note.created_at:%Y-%m-%d}] {note.author}: {note.body}  {list(note.tags)}')
    if not detail.notes:
        print('no notes on this event')

## Export surfaces

`to_records` / `to_csv` / `to_jsonl` are always available. `to_pandas` /
`to_polars` / `write_parquet` need the `research` extra (pulled in by the
`notebooks` extra). Every format uses the same record shape.

In [ ]:
print(to_csv(cohort.rows)[:400])

In [ ]:
print(to_jsonl(cohort.rows)[:400])

In [ ]:
pl_df = to_polars(cohort.rows)
pl_df.select(['symbol', 'earnings_date', 'reaction_class', 'retention']).head()

In [ ]:
out = write_parquet(cohort.rows, 'cohort_snapshot.parquet')
print('wrote', out, out.stat().st_size, 'bytes')

In [ ]:
await pool.close()